In [1]:
%%capture
!pip install -U unsloth
!pip install --no-deps trl peft accelerate bitsandbytes datasets

In [2]:
import torch
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

GPU: Tesla T4
VRAM GB: 15.6


In [3]:
!git clone https://github.com/Hetul803/speakup.git
%cd speakup/finetune

Cloning into 'speakup'...
remote: Enumerating objects: 205, done.
remote: Counting objects: 100% (205/205), done.
remote: Compressing objects: 100% (131/131), done.
remote: Total 205 (delta 95), reused 178 (delta 68), pack-reused 0 (from 0)
Receiving objects: 100% (205/205), 6.67 MiB | 23.54 MiB/s, done.
Resolving deltas: 100% (95/95), done.
/content/speakup/finetune


In [4]:
 import json
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

MODEL_NAME = 'unsloth/gemma-4-E2B-it-unsloth-bnb-4bit'
MAX_SEQ_LENGTH = 1024
OUTPUT_DIR = './speakup-gemma4-t4-lora'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
    use_rslora=True,
)

def format_conversation(example):
    text = ''
    for msg in example['messages']:
        if msg['role'] == 'system':
            text += f"<start_of_turn>user\n[System: {msg['content']}]\n"
        elif msg['role'] == 'user':
            text += f"{msg['content']}<end_of_turn>\n<start_of_turn>model\n"
        elif msg['role'] == 'assistant':
            text += f"{msg['content']}<end_of_turn>\n"
    return {'text': text}

rows = [json.loads(line) for line in open('dataset/aac_training.jsonl')]
dataset = Dataset.from_list(rows).map(format_conversation, remove_columns=list(rows[0].keys()))
print('Training examples:', len(dataset))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Map:   0%|          | 0/61 [00:00<?, ? examples/s]

Training examples: 61


In [5]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='linear',
        output_dir=OUTPUT_DIR,
        report_to='none',
        save_strategy='epoch',
    ),
)
trainer.train()
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Saved adapter:', OUTPUT_DIR)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/61 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 61 | Num Epochs = 3 | Total steps = 24
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 15,519,744 of 5,138,697,760 (0.30% trained)


Step,Training Loss
1,0.990952
2,0.990927
3,0.983429
4,0.877712
5,0.737735
6,0.638632
7,0.531146
8,0.685908
9,0.355409
10,0.319560


Unsloth: Restored added_tokens_decoder metadata in ./speakup-gemma4-t4-lora/checkpoint-8/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./speakup-gemma4-t4-lora/checkpoint-16/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./speakup-gemma4-t4-lora/checkpoint-24/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./speakup-gemma4-t4-lora/tokenizer_config.json.


Saved adapter: ./speakup-gemma4-t4-lora


In [7]:
FastLanguageModel.for_inference(model)

prompt = """<start_of_turn>user
Communicator points at a cup and makes a soft mmm sound. Memory says soft mmm + cup means water. Return SpeakUp JSON.<end_of_turn>
<start_of_turn>model
"""

# Gemma 4 tokenizer is a Processor, so pass text= explicitly
inputs = tokenizer(
    text=[prompt],
    return_tensors="pt",
    padding=True,
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=220,
    do_sample=True,
    temperature=0.2,
    top_p=0.9,
)

print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])

<start_of_turn>user
Communicator points at a cup and makes a soft mmm sound. Memory says soft mmm + cup means water. Return SpeakUp JSON.<end_of_turn>
<start_of_turn>model
{"intent":"I want water","confidence":0.98,"spoken_phrase":"I want water please.","explanation":"Communicator points at cup and makes soft mmm sound. Memory confirms this is water.","alternatives":["I want water please.","Water, please.","I want something to drink"],"needs_confirmation":false,"urgency":"normal","emotion_detected":"neutral","explanation_for_urgency":"Communicator is making a soft sound, not urgent.","alternatives_for_urgency":["I want water now.","I want water please.","I want something to drink"],"needs_confirmation":false,"urgency":"normal","emotion_detected":"neutral","explanation_for_emotion":"Soft mmm sound is neutral.","alternatives_for_emotion":["I want water please.","I want water now.","I want something to drink"],"needs_urgency":"false","explanation_for_urgency":"Soft mmm sound is neutral.",

In [8]:
!zip -r speakup-gemma4-t4-lora.zip speakup-gemma4-t4-lora
from google.colab import files
files.download('speakup-gemma4-t4-lora.zip')

  adding: speakup-gemma4-t4-lora/ (stored 0%)
  adding: speakup-gemma4-t4-lora/checkpoint-16/ (stored 0%)
  adding: speakup-gemma4-t4-lora/checkpoint-16/training_args.bin (deflated 53%)
  adding: speakup-gemma4-t4-lora/checkpoint-16/scaler.pt (deflated 64%)
  adding: speakup-gemma4-t4-lora/checkpoint-16/rng_state.pth (deflated 26%)
  adding: speakup-gemma4-t4-lora/checkpoint-16/adapter_config.json (deflated 58%)
  adding: speakup-gemma4-t4-lora/checkpoint-16/chat_template.jinja (deflated 82%)
  adding: speakup-gemma4-t4-lora/checkpoint-16/trainer_state.json (deflated 71%)
  adding: speakup-gemma4-t4-lora/checkpoint-16/processor_config.json (deflated 69%)
  adding: speakup-gemma4-t4-lora/checkpoint-16/scheduler.pt (deflated 61%)
  adding: speakup-gemma4-t4-lora/checkpoint-16/README.md (deflated 65%)
  adding: speakup-gemma4-t4-lora/checkpoint-16/tokenizer_config.json (deflated 86%)
  adding: speakup-gemma4-t4-lora/checkpoint-16/adapter_model.safetensors (deflated 22%)
  adding: speakup-

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>